## 🎯 Learning Objectives
* Understand the key metrics for evaluating agentic RAG systems: faithfulness, relevance, and completeness.
* Learn how to implement programmatic evaluation techniques using an LLM-as-a-judge approach.
* Analyze the trade-offs, practical considerations, and modern use cases for evaluating complex agentic RAG workflows.


# Evaluating Agentic RAG: Faithfulness, Relevance, and Completeness

As AI search engineers, building state-of-the-art Retrieval Augmented Generation (RAG) systems, especially those with agentic capabilities, demands rigorous evaluation. Unlike traditional RAG, agentic RAG introduces dynamic decision-making, multi-step reasoning, and tool use, making evaluation both more critical and more complex. We need to ensure our agents are not just generating text, but generating *reliable*, *pertinent*, and *thorough* information.

Think of your agentic RAG system as a highly skilled detective investigating a complex case based on a client's query. To assess the detective's performance, you'd ask three fundamental questions:

1.  **Faithfulness (or Groundedness): Did the detective stick to the facts?**
    *   This metric measures whether the generated answer is *fully supported* by the retrieved context. In the world of RAG, it's our primary defense against **hallucinations**. An unfaithful answer might sound plausible but contains information not present in the source documents. For agentic RAG, this is paramount, as an unfaithful statement early in a multi-step reasoning process can lead to cascading errors.

2.  **Relevance (or Answer Relevance): Did the detective answer the client's question directly?**
    *   This metric assesses how well the generated answer addresses the user's original query. An answer might be factually correct (faithful) but completely miss the point of the question. For agentic systems, where the agent might choose different tools or paths, ensuring the final output remains relevant to the *initial* user intent is crucial.

3.  **Completeness (or Context Recall/Retrieval Completeness): Did the detective gather and use all necessary evidence?**
    *   This metric has two facets: first, whether the *retrieved context itself* contained all the necessary information to answer the query; and second, whether the *generated answer utilized all the pertinent information* from that context. For agentic RAG, this extends to evaluating if the agent's dynamic process successfully identified and leveraged all critical pieces of information across its tool uses and reasoning steps to form a comprehensive answer.

### Why These Metrics Matter for Agentic RAG (2026 Perspective)

In 2026, with LLMs becoming increasingly powerful and agentic frameworks like LangGraph enabling sophisticated reasoning, the stakes for evaluation are higher. Agentic systems can dynamically modify their search queries, chain multiple tools, and even self-correct. This flexibility, while powerful, also introduces more opportunities for errors to creep in:

*   **Propagating Errors:** A slight hallucination or irrelevant piece of information in an intermediate step can derail the entire agent's reasoning path.
*   **Tool Misuse:** An agent might retrieve too much, too little, or irrelevant information if its tool-use strategy isn't optimal, impacting completeness and relevance.
*   **Complex Reasoning Chains:** Evaluating the final answer alone might not reveal where an agent went wrong in a multi-hop reasoning process. These metrics help pinpoint issues.

Modern evaluation often leverages **LLM-as-a-Judge** frameworks, where a separate, powerful LLM is prompted to act as an impartial evaluator, scoring and providing reasons for its assessment. This approach, combined with synthetic data generation and specialized evaluation platforms, forms the backbone of robust RAG evaluation in today's advanced AI landscape.


In [ ]:
import os
# In a real scenario, you might use actual LLM APIs. For demonstration, we'll use a mock.
# from langchain_openai import ChatOpenAI
# from langchain_core.prompts import ChatPromptTemplate
# from langchain_core.output_parsers import StrOutputParser

# --- Mock LLM for Demonstration Purposes ---
# This class simulates an LLM's behavior without requiring API keys.
# In a production setting, you would replace this with an actual LLM client (e.g., OpenAI, Anthropic, Gemini).
class MockLLM:
    def __init__(self, responses):
        self.responses = responses
        self.call_count = 0

    def invoke(self, prompt):
        # Simulate LLM thinking and returning a score/reason based on predefined responses.
        # This allows the example to be runnable without external dependencies.
        response_key = f"call_{self.call_count}"
        self.call_count += 1
        # Fallback for unexpected calls, though our example is designed to match.
        return self.responses.get(response_key, "Score: 3/5. Reason: Generic response from mock LLM.")

# --- Simplified Agentic RAG Components ---
# These classes represent a basic RAG system to generate answers for evaluation.
class Retriever:
    """Simulates retrieving documents based on a query."""
    def retrieve(self, query):
        # In a real system, this would involve vector databases, keyword search, etc.
        if "LangGraph" in query.lower():
            return [
                "LangGraph is a library for building stateful, multi-actor applications with LLMs.",
                "It extends LangChain to enable cyclical graphs and more complex agentic workflows.",
                "LangGraph allows for defining nodes and edges, creating flexible computational graphs."
            ]
        elif "evaluation metrics" in query.lower() or "evaluate RAG" in query.lower():
            return [
                "RAG evaluation metrics include faithfulness, relevance, and completeness.",
                "Faithfulness checks if the answer is grounded in context, preventing hallucinations.",
                "Relevance checks if the answer directly addresses the user's query.",
                "Completeness checks if all necessary information from the context was used and if the context was sufficient."
            ]
        else:
            return [
                "General knowledge about AI and LLMs.",
                "Some unrelated document about machine learning models."
            ]

class Generator:
    """Simulates an LLM generating an answer based on retrieved context."""
    def generate(self, query, context):
        context_str = "\n".join(context)
        # A very simple heuristic for generation; a real LLM would be used here.
        if "LangGraph" in query.lower() and "LangGraph is a library" in context_str:
            return "LangGraph is a powerful library built on LangChain, designed for creating complex, stateful LLM applications, especially those with cyclical reasoning patterns and multi-actor coordination."
        elif ("evaluation metrics" in query.lower() or "evaluate RAG" in query.lower()) and "Faithfulness checks" in context_str:
            return "Evaluating RAG systems involves assessing faithfulness (groundedness in context), relevance (addressing the query), and completeness (utilization of all pertinent context and sufficiency of context)."
        elif "quantum computing" in query.lower() and "General knowledge about AI" in context_str:
            # Simulate a generic, unhelpful answer due to irrelevant context
            return "I can provide general information about AI and machine learning, but I don't have specific details on quantum computing from the provided context."
        else:
            return "I'm sorry, I couldn't generate a specific answer based on the provided context."

class AgenticRAGSystem:
    """Orchestrates retrieval and generation, simulating a basic agentic flow."""
    def __init__(self, retriever, generator):
        self.retriever = retriever
        self.generator = generator

    def run(self, query):
        # In a real agentic system, this 'run' method would involve more complex
        # reasoning, tool selection, and iterative steps (e.g., using LangGraph).
        retrieved_docs = self.retriever.retrieve(query)
        generated_answer = self.generator.generate(query, retrieved_docs)
        return {"query": query, "context": retrieved_docs, "answer": generated_answer}

# --- Evaluation Functions (LLM-as-a-Judge Approach) ---
# These functions use an LLM (our MockLLM here) to score the RAG output.
# In 2026, dedicated evaluation frameworks like Ragas or LlamaIndex's evaluation modules
# often abstract these prompts, but the underlying principle is the same.

def evaluate_faithfulness(query, context, answer, evaluator_llm):
    """Evaluates if the answer is fully supported by the provided context."""
    context_str = "\n".join(context)
    prompt = f"""
    You are an expert evaluator. Your task is to assess the faithfulness of a generated answer to the provided context.
    Faithfulness means the answer contains *only* information directly supported by the context. Do not use external knowledge.

    Query: {query}
    Context: {context_str}
    Answer: {answer}

    Is the answer fully supported by the context? Provide a score from 1 (not supported at all) to 5 (fully supported) and a brief reason.
    Format: Score: [1-5]. Reason: [Your reason].
    """
    response = evaluator_llm.invoke(prompt)
    return response

def evaluate_relevance(query, answer, evaluator_llm):
    """Evaluates how well the generated answer addresses the user's query."""
    prompt = f"""
    You are an expert evaluator. Your task is to assess the relevance of a generated answer to the user's query.
    Relevance means the answer directly addresses the query and is not off-topic.

    Query: {query}
    Answer: {answer}

    How relevant is the answer to the query? Provide a score from 1 (not relevant at all) to 5 (highly relevant) and a brief reason.
    Format: Score: [1-5]. Reason: [Your reason].
    """
    response = evaluator_llm.invoke(prompt)
    return response

def evaluate_completeness(query, context, answer, evaluator_llm):
    """Evaluates if the answer fully utilizes necessary information from the context and if the context was sufficient."""
    context_str = "\n".join(context)
    prompt = f"""
    You are an expert evaluator. Your task is to assess the completeness of the generated answer with respect to the provided context and query.
    Completeness means the answer utilizes all *necessary* information from the context to fully address the query. It also implies the context itself was sufficient to answer the query.

    Query: {query}
    Context: {context_str}
    Answer: {answer}

    Does the answer fully utilize the necessary information from the context to address the query, and was the context sufficient? Provide a score from 1 (missing crucial information or context insufficient) to 5 (fully complete) and a brief reason.
    Format: Score: [1-5]. Reason: [Your reason].
    """
    response = evaluator_llm.invoke(prompt)
    return response

# --- Main Execution --- 
if __name__ == "__main__":
    # Initialize our RAG system components
    retriever = Retriever()
    generator = Generator()
    agentic_rag = AgenticRAGSystem(retriever, generator)

    # Initialize a mock evaluator LLM with predefined responses for demonstration.
    # In a real application, this would be a call to a powerful LLM like GPT-4o, Gemini 1.5 Pro, Claude 3 Opus, etc.
    evaluator_responses = {
        "call_0": "Score: 5/5. Reason: The answer accurately describes LangGraph's purpose and features, directly supported by the context.",
        "call_1": "Score: 5/5. Reason: The answer directly and comprehensively addresses the query about LangGraph's usage.",
        "call_2": "Score: 5/5. Reason: The answer effectively synthesizes the key points from the context to provide a complete overview of LangGraph.",

        "call_3": "Score: 5/5. Reason: The answer correctly lists and briefly defines the three key evaluation metrics as found in the context.",
        "call_4": "Score: 5/5. Reason: The answer is highly relevant, directly answering the question about RAG evaluation metrics.",
        "call_5": "Score: 5/5. Reason: The answer fully utilizes the provided context to explain the RAG evaluation metrics, and the context was sufficient.",

        "call_6": "Score: 1/5. Reason: The answer is generic and not supported by the provided context, which is about general AI, not quantum computing. This indicates a hallucination or lack of grounding.",
        "call_7": "Score: 1/5. Reason: The answer does not address the query about quantum computing at all, indicating severe irrelevance.",
        "call_8": "Score: 1/5. Reason: The answer is incomplete because the context itself was irrelevant and did not contain information about quantum computing, making it impossible to provide a complete answer."
    }
    evaluator_llm = MockLLM(evaluator_responses)

    print("### Evaluating Agentic RAG System Performance ###\n")

    # --- Test Case 1: Good RAG performance for a specific query ---
    print("--- Test Case 1: LangGraph Definition (Expected Good Performance) ---")
    query1 = "What is LangGraph used for?"
    result1 = agentic_rag.run(query1)
    print(f"Query: {result1['query']}")
    print(f"Context: {result1['context']}")
    print(f"Answer: {result1['answer']}")

    faithfulness_score1 = evaluate_faithfulness(result1['query'], result1['context'], result1['answer'], evaluator_llm)
    relevance_score1 = evaluate_relevance(result1['query'], result1['answer'], evaluator_llm)
    completeness_score1 = evaluate_completeness(result1['query'], result1['context'], result1['answer'], evaluator_llm)

    print(f"  Faithfulness: {faithfulness_score1}")
    print(f"  Relevance: {relevance_score1}")
    print(f"  Completeness: {completeness_score1}")
    print("\n")

    # --- Test Case 2: Another good RAG performance for evaluation metrics ---
    print("--- Test Case 2: RAG Evaluation Metrics (Expected Good Performance) ---")
    query2 = "What are the key evaluation metrics for RAG systems?"
    result2 = agentic_rag.run(query2)
    print(f"Query: {result2['query']}")
    print(f"Context: {result2['context']}")
    print(f"Answer: {result2['answer']}")

    faithfulness_score2 = evaluate_faithfulness(result2['query'], result2['context'], result2['answer'], evaluator_llm)
    relevance_score2 = evaluate_relevance(result2['query'], result2['answer'], evaluator_llm)
    completeness_score2 = evaluate_completeness(result2['query'], result2['context'], result2['answer'], evaluator_llm)

    print(f"  Faithfulness: {faithfulness_score2}")
    print(f"  Relevance: {relevance_score2}")
    print(f"  Completeness: {completeness_score2}")
    print("\n")

    # --- Test Case 3: Poor RAG performance due to irrelevant context ---
    print("--- Test Case 3: Irrelevant Context (Expected Poor Performance) ---")
    query3 = "Tell me about quantum computing."
    result3 = agentic_rag.run(query3) # Retriever will return general AI docs, not quantum computing.
    print(f"Query: {result3['query']}")
    print(f"Context: {result3['context']}")
    print(f"Answer: {result3['answer']}")

    faithfulness_score3 = evaluate_faithfulness(result3['query'], result3['context'], result3['answer'], evaluator_llm)
    relevance_score3 = evaluate_relevance(result3['query'], result3['answer'], evaluator_llm)
    completeness_score3 = evaluate_completeness(result3['query'], result3['context'], result3['answer'], evaluator_llm)

    print(f"  Faithfulness: {faithfulness_score3}")
    print(f"  Relevance: {relevance_score3}")
    print(f"  Completeness: {completeness_score3}")
    print("\n")


### Interpreting the Output and Practical Considerations

The code above demonstrates a programmatic way to evaluate agentic RAG systems using an "LLM-as-a-Judge" approach. Let's break down how to interpret the results and the real-world implications.

#### Interpreting the Scores

Each evaluation function (`evaluate_faithfulness`, `evaluate_relevance`, `evaluate_completeness`) returns a string containing a score (1-5) and a reason. This structured output is crucial:

*   **Scores (1-5):**
    *   **5:** Excellent performance. The answer fully meets the criteria for that metric.
    *   **4:** Good performance, minor issues or slight room for improvement.
    *   **3:** Acceptable, but with noticeable flaws. Needs attention.
    *   **2:** Poor performance. Significant issues.
    *   **1:** Very poor performance. Fails completely on the metric.

*   **Reason:** The textual reason provided by the evaluator LLM is often more valuable than the score itself. It pinpoints *why* a score was given, offering actionable insights for debugging and improvement. For instance, a low faithfulness score might be accompanied by a reason like "The answer mentions X, which is not present in the context," directly telling you where the hallucination occurred.

In our example, Test Cases 1 and 2 show high scores across all metrics, indicating a well-performing RAG system for those specific queries. Test Case 3, however, demonstrates a failure due to irrelevant context, leading to low scores across the board, especially for relevance and completeness, as the system couldn't provide a meaningful answer.

#### Performance Trade-offs and Challenges

While LLM-as-a-Judge is powerful, it comes with its own set of trade-offs:

1.  **Cost and Latency:** Using a powerful LLM (like GPT-4o, Gemini 1.5 Pro, or Claude 3 Opus) for every evaluation can be expensive and slow, especially when evaluating large datasets (thousands or millions of Q&A pairs). This necessitates strategies like sampling, using smaller, fine-tuned evaluation models, or batching requests.
2.  **Subjectivity and Bias:** Even expert LLMs can exhibit biases or inconsistencies. Their judgments are sensitive to prompt engineering, the specific LLM model used, and the quality of the evaluation prompts. Human evaluation remains the gold standard but is not scalable.
3.  **Reproducibility:** LLM outputs can be non-deterministic. Setting temperature to 0 and using consistent prompts helps, but perfect reproducibility can be challenging.
4.  **Complexity for Agentic Workflows:** Evaluating multi-step agentic reasoning is harder. You might need to evaluate intermediate steps (e.g., tool selection, sub-query generation) in addition to the final answer. This requires more sophisticated evaluation prompts and potentially a graph-based evaluation approach.
5.  **Ground Truth Data:** While LLM-as-a-Judge reduces the need for extensive human-labeled ground truth answers, you still need a diverse and representative set of queries and contexts to test your system effectively. Synthetic data generation, often guided by evaluation metrics, helps create these test sets.

#### Typical Use Cases in 2026

These evaluation techniques are integral to the modern RAG development lifecycle:

*   **Automated CI/CD Pipelines:** Integrate evaluation into your continuous integration/continuous deployment (CI/CD) workflows. Any code change that degrades RAG performance (e.g., lower faithfulness scores) can automatically trigger alerts or block deployments.
*   **A/B Testing and Experimentation:** Compare different RAG configurations (e.g., new retrieval algorithms, different LLM generators, updated prompt templates, new tools for agents) by running them against a test set and comparing their evaluation scores.
*   **Production Monitoring:** Continuously monitor the performance of your live RAG systems. A sudden drop in relevance or faithfulness scores could indicate issues with data freshness, model drift, or changes in user query patterns.
*   **Debugging Agentic Workflows:** When an agentic RAG system fails, these metrics can help pinpoint the exact stage where the error occurred. Was the context retrieved irrelevant (affecting completeness/relevance)? Did the generator hallucinate despite good context (affecting faithfulness)?
*   **Synthetic Data Generation and Refinement:** Use evaluation metrics to filter and improve synthetically generated Q&A pairs. For example, only keep synthetic data where the generated answer is highly faithful to the synthetic context.

By systematically applying these evaluation metrics, AI search engineers can build more robust, reliable, and trustworthy agentic RAG systems.


### Resources for Advanced RAG Evaluation

To dive deeper into evaluating agentic RAG systems and explore more sophisticated tools, consider the following resources:

*   **LangGraph Documentation:**
    *   [LangGraph Official Documentation](https://langchain-ai.github.io/langgraph/)
    *   Explore examples of building complex agentic workflows that require robust evaluation.

*   **Ragas - RAG Assessment Framework:**
    *   [Ragas GitHub Repository](https://github.com/explodinggradients/ragas)
    *   [Ragas Documentation](https://docs.ragas.io/en/latest/)
    *   A leading framework specifically designed for RAG evaluation, offering metrics like faithfulness, answer relevance, context recall, and more, often leveraging LLM-as-a-judge.

*   **LlamaIndex Evaluation Modules:**
    *   [LlamaIndex Evaluation Guide](https://docs.llamaindex.ai/en/stable/module_guides/evaluating/root.html)
    *   LlamaIndex provides comprehensive tools for evaluating various aspects of RAG pipelines, including custom evaluation metrics and datasets.

*   **LangChain Evaluation Modules:**
    *   [LangChain Evaluation Documentation](https://python.langchain.com/docs/guides/evaluation/)
    *   LangChain offers built-in evaluation chains and datasets to assess LLM and RAG performance.

*   **Papers on LLM-as-a-Judge:**
    *   "Judging LLM-as-a-Judge with MT-Bench and Chatbot Arena" ([arXiv link](https://arxiv.org/abs/2306.05685))
    *   Explore research on the efficacy and limitations of using LLMs for evaluation.

*   **Google AI Studio / Gemini API:**
    *   [Google AI Studio](https://aistudio.google.com/)
    *   [Gemini API Documentation](https://ai.google.dev/docs/gemini_api_overview)
    *   For implementing actual LLM-as-a-judge evaluators, you'll need access to powerful models like Gemini 1.5 Pro.

*   **Hugging Face Datasets:**
    *   [Hugging Face Datasets Library](https://huggingface.co/docs/datasets/index)
    *   Find and share datasets for RAG evaluation, or use the library to manage your own synthetic and human-labeled evaluation data.
